In [ ]:
!pip install transformers peft trl datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
pip install dataset

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.39
    Uninstalling SQLAlchemy-2.0.39:
      Successfully uninstalled SQLAlchemy-2.0.39
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython-sql 0.5.0 requires sqlalchemy>=2.0, but you have sqlalchemy 1.4.54 which is incompatible.


In [ ]:
import os
import json
import re
import time
import torch
import sys  # ✅ Import sys to handle unwanted arguments
from dataclasses import dataclass, field
from typing import Optional

from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, HfArgumentParser, BitsAndBytesConfig, DataCollatorWithPadding
from trl import DPOConfig, DPOTrainer

# ✅ Mount Google Drive (if using Drive for storage)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 🔹 Define Base Directories
shared_drive_base = "/content/drive/MyDrive/THESIS/DPO_Trainer"
datasets_dir = f"{shared_drive_base}/Datasets"
model_dir = f"{shared_drive_base}/Models"

# 🔹 Define Paths
JSON_DATA_PATH = f"{datasets_dir}/CompiledET.json"
MULTI_LANG_MODEL = f"{model_dir}/PestAway_Final_QWEN_SFT_Epoch4"
new_model = f"{model_dir}/PestAway_Final_QWEN_DPO_E1"

# 🔹 Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MULTI_LANG_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MULTI_LANG_MODEL,
    torch_dtype=torch.float16,  # Adjust dtype based on Colab GPU
    low_cpu_mem_usage=True
)
model.config.use_cache = False

@dataclass
class ScriptArguments:
    beta: Optional[float] = field(default=0.1)
    model_name_or_path: Optional[str] = field(default=MULTI_LANG_MODEL)
    learning_rate: Optional[float] = field(default=5e-6)
    per_device_train_batch_size: Optional[int] = field(default=4)  # Reduce batch size for Colab
    gradient_accumulation_steps: Optional[int] = field(default=4)  # Adjust for Colab VRAM
    warmup_steps: Optional[int] = field(default=500)
    max_grad_norm: Optional[float] = field(default=1.0)
    lora_alpha: Optional[float] = field(default=16)
    lora_dropout: Optional[float] = field(default=0.1)
    lora_r: Optional[int] = field(default=64)
    max_prompt_length: Optional[int] = field(default=64)
    max_length: Optional[int] = field(default=512)
    num_train_epochs: Optional[int] = field(default=2)
    output_dir: Optional[str] = field(default="./dpo_results")

# 🔹 Load and process dataset
def load_and_process_data(data_path: str) -> Dataset:
    dataset = load_dataset('json', data_files=data_path, split='train')

    def process_data(sample):
        if "rejected" not in sample:
            sample["rejected"] = "No response provided"
        return {"prompt": sample["prompt"], "chosen": sample["chosen"], "rejected": sample["rejected"]}

    return dataset.map(process_data, remove_columns=dataset.column_names)

# 🔹 Model evaluation function
def evaluate_model_with_prompt(model, tokenizer, prompt: str):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt", max_length=64, truncation=True).input_ids.to(model.device)

    with torch.no_grad():
        output = model.generate(
            inputs,
            max_new_tokens=256,
            pad_token_id=tokenizer.pad_token_id,
            no_repeat_ngram_size=3,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    generated_answer = generated_text[len(prompt):].strip()
    sentences = re.split(r'[.?!]', generated_answer)
    meaningful_answer = sentences[0].strip() if sentences else generated_answer
    print(f"Question: {prompt}\nAnswer: {meaningful_answer}")

# ✅ Run the script
if __name__ == "__main__":
    # ✅ Fix for Colab's extra arguments
    sys.argv = [sys.argv[0]]  # Remove unwanted Jupyter-related arguments

    parser = HfArgumentParser(ScriptArguments)
    script_args = parser.parse_args_into_dataclasses()[0]  # ✅ Fixed argument parsing

    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

    model = AutoModelForCausalLM.from_pretrained(
        script_args.model_name_or_path,
        quantization_config=bnb_config,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,  # Adjust for Colab's GPU
    )
    model.config.use_cache = False
    tokenizer.pad_token = tokenizer.eos_token

    # ✅ Load dataset
    train_dataset = load_and_process_data(JSON_DATA_PATH)
    train_dataset = train_dataset.filter(lambda x: len(x["prompt"]) + len(x["chosen"]) <= script_args.max_length)

    # ✅ LoRA Config
    peft_config = LoraConfig(
        r=script_args.lora_r,
        lora_alpha=script_args.lora_alpha,
        lora_dropout=script_args.lora_dropout,
        target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc_in", "fc_out", "wte"],
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, peft_config)

    # ✅ Training arguments
    training_args = DPOConfig(
        beta=script_args.beta,
        max_prompt_length=script_args.max_prompt_length,
        max_length=script_args.max_length,
        output_dir=script_args.output_dir,
        per_device_train_batch_size=script_args.per_device_train_batch_size,
        gradient_accumulation_steps=script_args.gradient_accumulation_steps,
        learning_rate=script_args.learning_rate,
        warmup_steps=script_args.warmup_steps,
        max_grad_norm=script_args.max_grad_norm,
        num_train_epochs=script_args.num_train_epochs
    )

    # ✅ Initialize trainer
    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        args=training_args,
        train_dataset=train_dataset,
        processing_class=AutoTokenizer.from_pretrained(MULTI_LANG_MODEL)
    )

    # ✅ Train the model
    dpo_trainer.train()

    # ✅ Merge LoRA into base model
    merged_model = model.merge_and_unload()

    # ✅ Save final model
    merged_model.save_pretrained(new_model)
    tokenizer.save_pretrained(new_model)

    print(f"✅ Training complete. Merged model saved to: {new_model}")

Mounted at /content/drive


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: maveric-magsaysay-eng (maveric-magsaysay-eng-university-of-santo-tomas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.638300
1000,0.346900
1500,0.151100
2000,0.083100
2500,0.063300
3000,0.053100
3500,0.046800
4000,0.042600
4500,0.037400
5000,0.040600


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:355: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


UnboundLocalError: cannot access local variable 'active_adapters' where it is not associated with a value